# MemGPT and Letta [Optional]

**MemGPT** (now evolved into **Letta**) treats LLM context like an operating system treats RAM: a **small working set** plus **paged** memory the agent manages explicitly.

```
  ┌─────────────────────────────────────────┐
  │  MAIN CONTEXT (limited — "RAM")         │
  │  • system persona                       │
  │  • core memory blocks (persona/human)   │
  │  • recent messages                      │
  └─────────────────┬───────────────────────┘
                    │ agent memory tools
        ┌───────────┴───────────┐
        ▼                       ▼
  RECALL MEMORY            ARCHIVAL MEMORY
  (conversation search)    (long-term vector store)
```

The agent **decides** what stays in core memory, what gets archived, and what to search — rather than passively growing the prompt forever.

This optional notebook:

1. Explains MemGPT / Letta concepts
2. Implements a **plain-Python MemGPT-style agent** for ShopCo Support (no Letta install required)
3. Maps tiers to LangGraph checkpointer + `BaseStore` patterns you may already use
4. Includes an optional cell for the real **Letta** SDK when you choose to install it

In [ ]:
"""
MemGPT-style memory lab — ShopCo Support (simulated).
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False
USE_LETTA_SDK = False  # set True only if `letta` package is installed

CORE_CHAR_LIMIT = 400


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


print("MemGPT simulation environment ready.")

---

## 1. The OS Memory Metaphor

| OS concept | MemGPT / Letta | ShopCo example |
|------------|----------------|----------------|
| RAM | **Main context** | Persona + core blocks + recent chat |
| Page file / swap | **Recall memory** | Searchable conversation history |
| Disk archive | **Archival memory** | Old facts, policies discussed months ago |
| `malloc` / `free` | Memory **tools** | `core_memory_append`, `archival_memory_search` |

The model is not given infinite context — it **manages** what fits in RAM.

---

## 2. Core Memory Blocks

Fixed slots in main context, always visible to the agent:

- **persona** — who the agent is
- **human** — facts about the current user
- **scratch** — working notes for this case

In [ ]:
@dataclass
class CoreMemory:
    persona: str = (
        "I am ShopCo Support — helpful, concise, cite policy IDs when relevant."
    )
    human: str = ""
    scratch: str = ""

    def render(self) -> str:
        return (
            f"<persona>\n{self.persona}\n</persona>\n"
            f"<human>\n{self.human or '(unknown)'}\n</human>\n"
            f"<scratch>\n{self.scratch or '(empty)'}\n</scratch>"
        )

    def char_count(self) -> int:
        return len(self.render())

    def append_human(self, text: str) -> None:
        self.human = (self.human + "\n" + text).strip()[:CORE_CHAR_LIMIT]

    def append_scratch(self, text: str) -> None:
        self.scratch = (self.scratch + "\n" + text).strip()[:CORE_CHAR_LIMIT]

    def replace_human(self, text: str) -> None:
        self.human = text[:CORE_CHAR_LIMIT]


core = CoreMemory()
print(core.render())

---

## 3. Recall Memory — Searchable Conversation History

In [ ]:
@dataclass
class RecallEntry:
    role: str
    content: str
    ts: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())


class RecallMemory:
    def __init__(self) -> None:
        self._entries: list[RecallEntry] = []

    def append(self, role: str, content: str) -> None:
        self._entries.append(RecallEntry(role=role, content=content))

    def search(self, query: str, top_k: int = 3) -> list[RecallEntry]:
        terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
        scored: list[tuple[int, RecallEntry]] = []
        for e in self._entries:
            hay = e.content.lower()
            score = sum(1 for t in terms if t in hay) if terms else 0
            if score:
                scored.append((score, e))
        scored.sort(key=lambda x: -x[0])
        return [e for _, e in scored[:top_k]]

    def recent(self, n: int = 4) -> list[RecallEntry]:
        return self._entries[-n:]


recall = RecallMemory()
recall.append("user", "My name is Alice. Order ORD-1001.")
recall.append("assistant", "Hi Alice, noted order ORD-1001.")
print("Search 'Alice':", [e.content for e in recall.search("Alice")])

---

## 4. Archival Memory — Long-Term Store

In [ ]:
@dataclass
class ArchivalRecord:
    record_id: str
    text: str
    tags: list[str]
    created_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())


class ArchivalMemory:
    def __init__(self) -> None:
        self._records: list[ArchivalRecord] = []

    def insert(self, text: str, tags: list[str] | None = None) -> str:
        rid = f"arch-{uuid.uuid4().hex[:8]}"
        self._records.append(ArchivalRecord(rid, text, tags or []))
        return rid

    def search(self, query: str, top_k: int = 3) -> list[ArchivalRecord]:
        terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
        scored: list[tuple[int, ArchivalRecord]] = []
        for r in self._records:
            hay = (r.text + " " + " ".join(r.tags)).lower()
            score = sum(1 for t in terms if t in hay) if terms else 0
            if score:
                scored.append((score, r))
        scored.sort(key=lambda x: -x[0])
        return [r for _, r in scored[:top_k]]


archival = ArchivalMemory()
archival.insert("Alice prefers email contact. Tier: pro.", tags=["alice", "preference"])
archival.insert("ORD-1001 return discussed 2026-07-01 — eligible within 30 days.", tags=["ORD-1001", "returns"])
print("Archival search 'return ORD-1001':", [r.text for r in archival.search("return ORD-1001")])

---

## 5. Memory Tools — Agent-Managed Paging

MemGPT agents call tools to move information between tiers.

In [ ]:
class MemoryTool(str, Enum):
    CORE_APPEND_HUMAN = "core_memory_append_human"
    CORE_REPLACE_HUMAN = "core_memory_replace_human"
    CORE_APPEND_SCRATCH = "core_memory_append_scratch"
    ARCHIVAL_INSERT = "archival_memory_insert"
    ARCHIVAL_SEARCH = "archival_memory_search"
    RECALL_SEARCH = "conversation_search"


@dataclass
class ToolResult:
    tool: str
    output: str


class MemGPTToolRouter:
    def __init__(self, core: CoreMemory, recall: RecallMemory, archival: ArchivalMemory) -> None:
        self.core = core
        self.recall = recall
        self.archival = archival
        self.log: list[ToolResult] = []

    def execute(self, tool: MemoryTool, **kwargs: Any) -> str:
        if tool == MemoryTool.CORE_APPEND_HUMAN:
            self.core.append_human(kwargs["text"])
            out = "Updated human core memory."
        elif tool == MemoryTool.CORE_REPLACE_HUMAN:
            self.core.replace_human(kwargs["text"])
            out = "Replaced human core memory."
        elif tool == MemoryTool.CORE_APPEND_SCRATCH:
            self.core.append_scratch(kwargs["text"])
            out = "Updated scratch core memory."
        elif tool == MemoryTool.ARCHIVAL_INSERT:
            rid = self.archival.insert(kwargs["text"], kwargs.get("tags", []))
            out = f"Archived as {rid}"
        elif tool == MemoryTool.ARCHIVAL_SEARCH:
            hits = self.archival.search(kwargs["query"])
            out = "\n".join(r.text for r in hits) or "(no archival hits)"
        elif tool == MemoryTool.RECALL_SEARCH:
            hits = self.recall.search(kwargs["query"])
            out = "\n".join(f"{h.role}: {h.content}" for h in hits) or "(no recall hits)"
        else:
            out = "unknown tool"
        self.log.append(ToolResult(tool.value, out))
        return out


router = MemGPTToolRouter(core, recall, archival)
print(router.execute(MemoryTool.ARCHIVAL_SEARCH, query="Alice preference"))

---

## 6. Offline Memory Manager — Decides Tool Calls

Stand-in for the LLM planner that chooses memory operations before answering.

In [ ]:
def plan_memory_ops(user_message: str) -> list[tuple[MemoryTool, dict[str, Any]]]:
    ops: list[tuple[MemoryTool, dict[str, Any]]] = []
    text = user_message

    name_m = re.search(r"my name is ([A-Za-z]+)", text, re.I)
    if name_m:
        ops.append((MemoryTool.CORE_REPLACE_HUMAN, {"text": f"Name: {name_m.group(1)}"}))

    oid = re.search(r"ORD-[0-9]{4}", text.upper())
    if oid:
        ops.append((MemoryTool.CORE_APPEND_SCRATCH, {"text": f"Active order: {oid.group(0)}"}))

    if "remember" in text.lower() or "prefer" in text.lower():
        ops.append((MemoryTool.ARCHIVAL_INSERT, {"text": text, "tags": ["preference"]}))

    if "last time" in text.lower() or "previously" in text.lower():
        ops.append((MemoryTool.ARCHIVAL_SEARCH, {"query": text}))
        ops.append((MemoryTool.RECALL_SEARCH, {"query": text}))

    if "return" in text.lower() and oid:
        ops.append((MemoryTool.ARCHIVAL_SEARCH, {"query": f"return {oid.group(0)}"}))

    return ops


print(plan_memory_ops("My name is Alice. Remember I prefer email. Order ORD-1001."))

---

## 7. MemGPT-Style Agent Loop

In [ ]:
@dataclass
class AgentTurnResult:
    user_message: str
    memory_ops: list[str]
    main_context_preview: str
    answer: str
    tool_log: list[ToolResult]


class MemGPTStyleAgent:
    def __init__(self) -> None:
        self.core = CoreMemory()
        self.recall = RecallMemory()
        self.archival = ArchivalMemory()
        self.router = MemGPTToolRouter(self.core, self.recall, self.archival)

    def step(self, user_message: str) -> AgentTurnResult:
        self.recall.append("user", user_message)
        op_trace: list[str] = []

        for tool, kwargs in plan_memory_ops(user_message):
            out = self.router.execute(tool, **kwargs)
            op_trace.append(f"{tool.value} → {out[:60]}")

        answer = self._compose_answer(user_message)
        self.recall.append("assistant", answer)

        recent = "\n".join(f"{e.role}: {e.content}" for e in self.recall.recent(4))
        main_context = f"{self.core.render()}\n\n<recent>\n{recent}\n</recent>"

        return AgentTurnResult(
            user_message=user_message,
            memory_ops=op_trace,
            main_context_preview=main_context[:500] + "...",
            answer=answer,
            tool_log=list(self.router.log),
        )

    def _compose_answer(self, user_message: str) -> str:
        q = user_message.lower()
        human = self.core.human
        if "return" in q:
            hits = self.archival.search("return")
            extra = hits[0].text if hits else "Returns within 30 days per policy."
            return f"Based on your profile ({human}) and records: {extra}"
        if "prefer" in q or "remember" in q:
            return "I've saved that preference to archival memory for future chats."
        if "status" in q or "order" in q:
            return f"Checking scratch notes: {self.core.scratch or 'no order yet'}"
        return f"Hello! Core human block: {human or 'please share your name'}."


agent = MemGPTStyleAgent()
r1 = agent.step("My name is Alice. Order ORD-1001.")
print("Turn 1:", r1.answer)
print("Ops:", r1.memory_ops)

---

## 8. Multi-Turn — Archival Recall Across Turns

In [ ]:
agent2 = MemGPTStyleAgent()
agent2.step("My name is Alice. Remember I prefer email contact.")
agent2.archival.insert(
    "ORD-1001 shipped 2026-07-01. Return eligible.",
    tags=["ORD-1001"],
)

r2 = agent2.step("Last time we discussed ORD-1001 — can I still return it?")
print("Turn 2:", r2.answer)
print("Memory ops:", r2.memory_ops)

---

## 9. What Is Letta?

**Letta** (formerly the MemGPT research stack open-sourced) is a platform for **stateful agents** with:

| Feature | Description |
|---------|-------------|
| **Persistent agents** | Agents survive restarts with memory blocks |
| **Memory blocks** | Editable persona / human / custom sections |
| **Archival + recall** | Same tiered model as MemGPT |
| **Tool use** | Standard tools plus memory management |
| **Sleep-time compute** | Background memory consolidation (advanced) |
| **Server + SDK** | REST API and Python client for deployment |

Conceptually, Letta **productizes** the simulation you ran above — with vector archival, hosted storage, and LLM-driven memory tool selection.

---

## 10. Map MemGPT Tiers to LangGraph Primitives

| MemGPT / Letta | LangGraph / DIY equivalent |
|----------------|----------------------------|
| Main context + core blocks | System prompt + state fields + recent `messages` |
| Recall memory | Checkpointer message history + `conversation_search` tool |
| Archival memory | `BaseStore` / vector DB + `archival_memory_search` tool |
| Memory tools in loop | ToolNode with memory write/search tools |
| Sleep-time compute | Background graph / cron summarization job |

You can build MemGPT-like behavior in LangGraph without Letta — Letta saves operational glue.

---

## 11. When to Use Letta vs Roll Your Own

| Choose **Letta** | Choose **LangGraph + store** |
|------------------|------------------------------|
| Fast POC for stateful agents | Existing LangGraph investment |
| Want memory tools out of the box | Custom memory governance |
| Agent hosting included | Your own infra / compliance |
| Experiment with sleep-time memory | Simpler thread + store suffices |

Many production teams use **checkpointer + BaseStore + compression** without adopting a full Letta deployment.

---

## 12. Core Memory Limits — Why Replace vs Append

Core blocks have **character limits**. When user facts change, use **replace** not append to avoid contradictory human blocks.

In [ ]:
demo_core = CoreMemory()
demo_core.append_human("Name: Alice. Tier: basic.")
demo_core.append_human("Tier upgraded to pro.")  # messy — two tiers
print("Append-only (messy):\n", demo_core.human)

demo_core.replace_human("Name: Alice. Tier: pro. Contact: email.")
print("\nReplace (clean):\n", demo_core.human)

---

## 13. Optional — Letta SDK (If Installed)

Set `USE_LETTA_SDK = True` only after installing the `letta` package in your environment. This cell fails gracefully when the SDK is absent.

In [ ]:
def try_letta_client() -> dict[str, Any]:
    try:
        from letta import create_client  # type: ignore
    except ImportError:
        return {"status": "letta_not_installed", "hint": "pip install letta to run this cell"}

    client = create_client()
    return {
        "status": "ok",
        "client_type": type(client).__name__,
        "note": "Create agent with memory blocks via client.create_agent(...)",
    }


if USE_LETTA_SDK:
    print(pretty(try_letta_client()))
else:
    print("Simulation mode — MemGPTStyleAgent demonstrated above.")
    print("Set USE_LETTA_SDK = True after installing letta to probe the SDK.")

---

## 14. Optional Live LLM Memory Planner

Set `USE_LIVE_LLM = True` to let `gpt-4o-mini` choose memory tool calls (experimental).

In [ ]:
def live_memory_plan(user_message: str) -> list[tuple[MemoryTool, dict[str, Any]]]:
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    tools_desc = [t.value for t in MemoryTool]
    resp = llm.invoke([
        SystemMessage(content=(
            f"Pick memory tools from {tools_desc} for this user message. "
            "Reply JSON list of {{tool, kwargs}}."
        )),
        HumanMessage(content=user_message),
    ])
    try:
        planned = json.loads(str(resp.content))
        return [(MemoryTool(p["tool"]), p.get("kwargs", {})) for p in planned]
    except (json.JSONDecodeError, KeyError, ValueError):
        return plan_memory_ops(user_message)


if USE_LIVE_LLM:
    print(live_memory_plan("Alice ORD-1001 return eligibility"))
else:
    print("Offline planner:", plan_memory_ops("Alice ORD-1001 return eligibility"))

---

## 15. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Unlimited core append | Contradictory human block | `replace` when facts update |
| No archival search before answer | "Forgot" old cases | Search archival on "last time" queries |
| Dump everything in main context | Context rot returns | Page to archival |
| Recall without checkpoint | Lost on restart | Persist recall store |
| Letta for simple FAQ bot | Overkill | RAG + thread memory enough |

---

## 16. Quiz

1. What are the three MemGPT memory tiers?
2. What is the difference between recall and archival memory?
3. Why do core memory blocks have size limits?
4. How does Letta relate to MemGPT?
5. What LangGraph primitives map to archival memory?

---

## 17. Summary

**MemGPT** pioneered LLM-managed memory paging: **core blocks** in main context, **recall** for conversation search, **archival** for long-term facts. **Letta** packages this as a deployable agent platform.

This notebook simulated the loop on ShopCo Support without external dependencies. The agent called memory tools, kept a bounded core, and searched archival records for return eligibility.

For many teams, **LangGraph checkpointer + BaseStore + compression** achieves similar goals — Letta is valuable when you want the full memory-tool agent host without building it yourself.